# Affinity Benchmark Timing

Scans every `affinity_timing_log.csv` under `data/results/affinity_benchmark/` and plots, per model, the distribution of number of abstracts evaluated (`targets_evaluated / 48`) vs `elapsed_seconds`. Each point is one logged run (a stage completing on a subset or all abstracts). A linear fit through the origin per model/stage gives an extrapolated seconds-per-abstract rate.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

CATEGORICAL_COLORS = [
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948",
    "#636efa", "#ef553b", "#00cc96", "#ab63fa", "#ffa15a", "#19d3f3", "#ff6692", "#b6e880"
]
FEW_SHOT_SYMBOLS = ["circle", "diamond", "square", "cross"]
FEW_SHOT_DASHES = ["solid", "dash", "dot", "dashdot"]

## Scan and load timing logs

In [ ]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

timing_log_paths = sorted(bench_root.rglob("affinity_timing_log.csv"))

timing_frames = []
for log_path in timing_log_paths:
    run_df = pd.read_csv(log_path)
    run_df["model"] = log_path.parent.name
    run_df["benchmark_run"] = log_path.parent.parent.name
    timing_frames.append(run_df)

timing_df = pd.concat(timing_frames, ignore_index=True)
timing_df["timestamp_utc"] = pd.to_datetime(timing_df["timestamp_utc"])

In [ ]:
timing_df.shape

In [ ]:
timing_df.head()

## Filters

- `MODELS_TO_INCLUDE`: only these models are kept.
- `STAGES_TO_INCLUDE`: subset of `["ra", "prp"]` — pick one or both.
- `targets_evaluated` must be a multiple of `top_k` (48, per `settings.yaml`); runs that don't line up with a full micro-batch pass are discarded.
- `BENCHMARK_RUNS_TO_INCLUDE`: only these `benchmark_run` folder names are kept; leave it empty (`[]` or `None`) to keep every run found under `affinity_benchmark/`.

In [ ]:
MODELS_TO_INCLUDE = [
    "gemini-3.1-flash-lite",
    "gemma4-31b-cloud",
    # "gpt-5.4",
    "gpt-5.4-mini",
    "mistral-3-14b-cloud",
    "mistral-large-3-675b-cloud",
    # "deepseek-v4-flash",
    "deepseek-v4-flash-cloud",
    # "deepseek-v4-flash-0731-cloud",
    "claude-haiku-4.5",
    "mistral-large-2512",
    "mistral-8b-2512",
    "mistral-small-2603",
    # "grok-4.20-0309-reasoning",
    "qwen3.6-flash"
]
STAGES_TO_INCLUDE = ["ra"]
TARGETS_EVALUATED_MULTIPLE_OF = 48
BENCHMARK_RUNS_TO_INCLUDE = [
    "20260724_143127",
    "20260724_155639",
    "20260726_134607",
    "20260726_134607_1",
    "20260726_134607_3",
    "20260726_134607_4",
    "20260726_134607_5",
    "20260726_154151",
    "20260726_154151_1",
    "20260726_154151_3",
    "20260726_154151_4",
    "20260731_135854",
    "20260804_120330",
    "tpwrs_2025_tpwrs_2025",
    "tpwrs_2026_tpwrs_2026"
]

timing_df = timing_df[timing_df["model"].isin(MODELS_TO_INCLUDE)]
timing_df = timing_df[timing_df["stage"].isin(STAGES_TO_INCLUDE)]
timing_df = timing_df[timing_df["targets_evaluated"] % TARGETS_EVALUATED_MULTIPLE_OF == 0]

if BENCHMARK_RUNS_TO_INCLUDE:
    timing_df = timing_df[timing_df["benchmark_run"].isin(BENCHMARK_RUNS_TO_INCLUDE)]
timing_df["num_abstracts"] = timing_df["targets_evaluated"] // TARGETS_EVALUATED_MULTIPLE_OF
timing_df['stage'] = timing_df['stage'].replace({'ra': 'Benchmark'})

## Recover `few_shot_max_examples_per_prompt` per benchmark run

Each benchmark run folder (e.g. `data/results/affinity_benchmark/20260726_154151/`) has a top-level
`run_metadata.json` recording the settings the run was launched with, including
`llm_reasoner.few_shot_max_examples_per_prompt`. If that parameter can be recovered for every
included benchmark run, it is added to the timing/fit grouping below (since it can materially change
the seconds-per-abstract rate); otherwise the fit falls back to grouping by `model`/`stage` only.

In [ ]:
import json

def load_few_shot_max_examples(bench_root, benchmark_run):
    run_dir = bench_root / benchmark_run
    json_files = list(run_dir.glob("*.json"))
    if not json_files:
        return None
    with open(json_files[0]) as f:
        metadata = json.load(f)
    return metadata["settings"]["runtime"]["llm_reasoner"].get("few_shot_max_examples_per_prompt")

few_shot_by_run = {
    benchmark_run: load_few_shot_max_examples(bench_root, benchmark_run)
    for benchmark_run in timing_df["benchmark_run"].unique()
}
few_shot_available_for_all_runs = all(value is not None for value in few_shot_by_run.values())

if few_shot_available_for_all_runs:
    timing_df["few_shot_max_examples_per_prompt"] = timing_df["benchmark_run"].map(few_shot_by_run)

print("few_shot_max_examples_per_prompt by benchmark_run:", few_shot_by_run)
print("Available for all included runs:", few_shot_available_for_all_runs)


## Per-model, per-stage linear fit (through origin)

`s_per_abstract_fit` is the total elapsed time divided by the total number of abstracts evaluated
across all logged runs for that group — a weighted average that extrapolates linearly from the
origin. The group is `model`/`stage`, further split by `few_shot_max_examples_per_prompt` whenever
that parameter was recoverable for every included benchmark run.

In [ ]:
GROUP_COLS = (
    ["model", "stage", "few_shot_max_examples_per_prompt"]
    if few_shot_available_for_all_runs
    else ["model", "stage"]
)

fit_df = (
    timing_df.groupby(GROUP_COLS)
    .agg(
        n_runs=("elapsed_seconds", "size"),
        total_abstracts=("num_abstracts", "sum"),
        total_seconds=("elapsed_seconds", "sum"),
    )
    .reset_index()
)
fit_df["s_per_abstract_fit"] = fit_df["total_seconds"] / fit_df["total_abstracts"]
fit_df.sort_values(["stage", "s_per_abstract_fit"]).reset_index(drop=True)

## Abstracts evaluated vs elapsed seconds, all models

Interactive Plotly scatter across all models: `color` groups by `model`, marker `symbol` (and,
for the fit lines, `line_dash`) groups by `few_shot_max_examples_per_prompt` when that parameter
is available for every included run, and each `stage` gets its own facet column. Lines show the
per-group linear fit through the origin from the previous cell.

In [ ]:
# STAGE_TO_PLOT = 'ra'

In [ ]:
def build_fit_line_df(fit_df, timing_df, group_cols):
    rows = []
    for _, fit_row in fit_df.iterrows():
        mask = pd.Series(True, index=timing_df.index)
        for col in group_cols:
            mask &= timing_df[col] == fit_row[col]
        x_max = timing_df.loc[mask, "num_abstracts"].max()
        for num_abstracts in (0, x_max):
            rows.append({
                **{col: fit_row[col] for col in group_cols},
                "num_abstracts": num_abstracts,
                "elapsed_seconds": fit_row["s_per_abstract_fit"] * num_abstracts,
            })
    return pd.DataFrame(rows)


fit_line_df = build_fit_line_df(fit_df, timing_df, GROUP_COLS)

model_color_map = dict(zip(MODELS_TO_INCLUDE, CATEGORICAL_COLORS))
few_shot_col = "few_shot_max_examples_per_prompt" if few_shot_available_for_all_runs else None
few_shot_values = sorted(fit_df[few_shot_col].unique()) if few_shot_col else []
symbol_map = {value: FEW_SHOT_SYMBOLS[i % len(FEW_SHOT_SYMBOLS)] for i, value in enumerate(few_shot_values)}
dash_map = {value: FEW_SHOT_DASHES[i % len(FEW_SHOT_DASHES)] for i, value in enumerate(few_shot_values)}

category_orders = {"stage": sorted(timing_df["stage"].unique())}

# timing_df = timing_df[timing_df["stage"] == STAGE_TO_PLOT]

if few_shot_col:
    category_orders[few_shot_col] = few_shot_values

scatter_fig = px.scatter(
    timing_df,
    x="num_abstracts",
    y="elapsed_seconds",
    color="model",
    # symbol=few_shot_col,
    facet_col=few_shot_col,
    color_discrete_map=model_color_map,
    symbol_map=symbol_map,
    category_orders=category_orders,
    labels={"num_abstracts": "abstracts evaluated", "elapsed_seconds": "elapsed seconds"},
)
scatter_fig.update_traces(marker_size=9)

line_fig = px.line(
    fit_line_df,
    x="num_abstracts",
    y="elapsed_seconds",
    color="model",
    # line_dash=few_shot_col,
    facet_col=few_shot_col,
    color_discrete_map=model_color_map,
    line_dash_map=dash_map,
    category_orders=category_orders,
)
line_fig.update_traces(showlegend=False, opacity=0.7)

fig = go.Figure(data=line_fig.data + scatter_fig.data, layout=scatter_fig.layout)
fig.update_layout(
    title="Abstracts evaluated vs elapsed seconds, all models",
    legend_title="model, few_shot" if few_shot_col else "model",
)
fig.show()

## Extrapolated time for an abstract count

Edit `EXTRAPOLATION_ABSTRACT_COUNT` to the number of abstracts you plan to score, then read off the estimated wall-clock time per model/stage.

In [ ]:
EXTRAPOLATION_ABSTRACT_COUNT = 32000

extrapolation_df = fit_df[GROUP_COLS + ["n_runs", "s_per_abstract_fit"]].copy()
extrapolation_df["estimated_hours"] = (
    EXTRAPOLATION_ABSTRACT_COUNT * extrapolation_df["s_per_abstract_fit"] / 3600
)
extrapolation_df.sort_values(["stage", "estimated_hours"]).reset_index(drop=True)

## Full-corpus publication volume and extrapolated compute time

Reads `data/processed/abstracts_cleaned_all_journals.csv`, counts publications per `publication_year` and `publication_journal`, then derives the cumulative number of publications when processing backwards in time (starting at the most recent year and working back to the earliest). Combined with the per-model/stage `s_per_abstract_fit` rates fit above, this extrapolates the wall-clock time to reach each year of the backlog.

In [ ]:
abstracts_path = repo_root / "data" / "processed" / "abstracts_cleaned.csv"
abstracts_df = pd.read_csv(abstracts_path)
abstracts_df.shape

### Publications per year and journal

In [ ]:
publications_by_year_journal = (
    abstracts_df.groupby(["publication_year", "publication_journal"])
    .size()
    .reset_index(name="num_publications")
)
publications_by_year_journal

### Cumulative publications, backwards in time (last year to first year)

Sums publications across all journals per year, then takes the cumulative sum starting from the last year and accumulating backwards to the first year — i.e. `cumulative_publications` at year `y` is the total publication count if processing started at the most recent year and worked back through year `y`.

In [ ]:
publications_by_year = (
    publications_by_year_journal.groupby("publication_year")["num_publications"]
    .sum()
    .sort_index(ascending=False)
)
cumulative_publications_df = (
    publications_by_year.cumsum()
    .sort_index()
    .reset_index(name="cumulative_publications")
)
cumulative_publications_df

### Extrapolated compute time to reach each year

For each `model`/`stage` (and `few_shot_max_examples_per_prompt` group, when available), multiplies `cumulative_publications` by the `s_per_abstract_fit` rate from the fit above to extrapolate the wall-clock time to process the backlog back to each year.

In [ ]:
compute_time_by_year_df = fit_df[GROUP_COLS + ["s_per_abstract_fit"]].merge(
    cumulative_publications_df, how="cross"
)
compute_time_by_year_df["elapsed_hours"] = (
    compute_time_by_year_df["cumulative_publications"] * compute_time_by_year_df["s_per_abstract_fit"] / 3600
)
compute_time_by_year_df["elapsed_days"] = compute_time_by_year_df["elapsed_hours"] / 24
# compute_time_by_year_df=compute_time_by_year_df[compute_time_by_year_df['stage'] == STAGE_TO_PLOT] 
compute_time_by_year_df

### Extrapolated elapsed time by year, all models

Same `color`/`facet_col`/`line_dash` structure as the abstracts-vs-elapsed-seconds plot above: `color` groups by `model`, `line_dash` groups by `few_shot_max_examples_per_prompt` when available, and each `stage` gets its own facet column. `x` is `publication_year` running from the first year to the last, `y` is the extrapolated elapsed hours to process the cumulative backlog back to that year.

In [ ]:
compute_time_fig = px.line(
    compute_time_by_year_df,
    x="publication_year",
    y="elapsed_days",
    color="model",
    # line_dash=few_shot_col,
    facet_col=few_shot_col,
    color_discrete_map=model_color_map,
    line_dash_map=dash_map,
    category_orders=category_orders,
    labels={"publication_year": "publication year", "elapsed_days": "extrapolated elapsed days"},
)
compute_time_fig.update_traces(mode="lines+markers", marker_size=6)
compute_time_fig.update_layout(
    title="Extrapolated compute time to process cumulative publications, backwards from last year",
    legend_title="model, few_shot" if few_shot_col else "model",
)
compute_time_fig.show()

In [ ]:
publications_by_year_journal = (
    abstracts_df.groupby(["publication_year", "publication_journal"])
    .size()
    .reset_index(name="num_publications")
)
publications_by_year_journal

## Cost

In [ ]:
import sys

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from scripts.util.cost_estimation_support import (
    COST_EXTRAPOLATION_ABSTRACT_COUNT,
    build_cost_estimation_df,
    calculate_api_cost_usd,
    MODEL_PRICING_USD,
    )

cost_estimation_df = build_cost_estimation_df()
cost_estimation_df

In [ ]:
cost_by_year_df = (
    cost_estimation_df[
        ["stage", "model", "few_shot_max_examples_per_prompt", "cost_per_abstract"]
    ]
    .merge(cumulative_publications_df, how="cross")
)

cost_by_year_df["estimated_cost"] = (
    cost_by_year_df["cumulative_publications"] * cost_by_year_df["cost_per_abstract"]
)

cost_by_year_df = cost_by_year_df.sort_values(
    ["stage", "model", "few_shot_max_examples_per_prompt", "publication_year"]
).reset_index(drop=True)

cost_by_year_df = cost_by_year_df[cost_by_year_df['stage'] != 'ra']
cost_by_year_df

In [ ]:
compute_cost_figure = px.bar(
    cost_by_year_df,
    x="publication_year",
    y="estimated_cost",
    color="model",
    # line_dash=few_shot_col,
    facet_col=few_shot_col,
    facet_row="stage",
    color_discrete_map=model_color_map,
    # line_dash_map=dash_map,
    category_orders=category_orders,
    labels={"publication_year": "publication year", "estimated_cost": "extrapolated cost $"},
)
# compute_cost_figure.update_traces(mode="stacked",)
compute_cost_figure.update_layout(
    title="Extrapolated compute cost to process cumulative publications, backwards from last year",
    legend_title="model"
)
compute_cost_figure.show()